In [ ]:
# Interlude: Attention as Test-Time Regression
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/interludes/attention-as-test-time-regression.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/interludes').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/interludes')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

import matplotlib.pyplot as plt
import numpy as np

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Verify softmax attention's local-constant optimum.
3. Report or visualize the measured result.

In [ ]:
# [1]
ms_rng = np.random.default_rng(6050)
ms_d, ms_T = 8, 12
ms_K = ms_rng.standard_normal((ms_T, ms_d))
ms_V = ms_rng.standard_normal((ms_T, 3))
ms_q = ms_rng.standard_normal(ms_d)
ms_scores = ms_K @ ms_q / np.sqrt(ms_d)
ms_kappa = np.exp(ms_scores - ms_scores.max())
ms_c = (ms_kappa[:, None] * ms_V).sum(0) / ms_kappa.sum()
ms_attention = (ms_kappa / ms_kappa.sum()) @ ms_V
ms_gradient = (ms_kappa[:, None] * (ms_c - ms_V)).sum(0)
# [2]
assert np.allclose(ms_c, ms_attention)
# [3]
print(f"max |stationarity gradient|: {np.abs(ms_gradient).max():.3e}")

**Plan**

1. Define the reusable `ms_phi` helper.
2. Prepare the inputs and fixed settings for the example.
3. Compare a factorized-kernel traversal with its running state.
4. Check the claimed identities, shapes, or invariants.
5. Report or visualize the measured result.

In [ ]:
# [1]
def ms_phi(x):
    return np.where(x > 0, x + 1, np.exp(x))

# [2]
ms_S = np.zeros((ms_d, ms_V.shape[1]))
ms_z = np.zeros(ms_d)
# [3]
for ms_key, ms_value in zip(ms_K, ms_V, strict=True):
    ms_S += np.outer(ms_phi(ms_key), ms_value)
    ms_z += ms_phi(ms_key)
ms_streaming = (ms_S.T @ ms_phi(ms_q)) / (ms_z @ ms_phi(ms_q))
ms_weights = ms_phi(ms_K) @ ms_phi(ms_q)
ms_traversal = (ms_weights[:, None] * ms_V).sum(0) / ms_weights.sum()
# [4]
assert np.allclose(ms_streaming, ms_traversal)
# [5]
print("streaming equals traversal:", np.allclose(ms_streaming, ms_traversal))
print(f"max |difference|: {np.abs(ms_streaming-ms_traversal).max():.3e}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Verify that one SGD step is the delta recurrence.
3. Report or visualize the measured result.

In [ ]:
# [1]
ms_eta = 0.3
ms_h = ms_rng.standard_normal(ms_d)
ms_k = ms_rng.standard_normal(ms_d)
ms_v = ms_rng.standard_normal()
ms_sgd = ms_h - ms_eta * ms_k * (ms_k @ ms_h - ms_v)
ms_delta = ((np.eye(ms_d) - ms_eta * np.outer(ms_k, ms_k)) @ ms_h
            + ms_eta * ms_v * ms_k)
# [2]
assert np.allclose(ms_sgd, ms_delta)
# [3]
print("SGD equals Delta recurrence:", np.allclose(ms_sgd, ms_delta))
print(f"max |difference|: {np.abs(ms_sgd-ms_delta).max():.3e}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable helpers: `ms_unit` and `ms_decode`.
3. Define the reusable helpers: `ms_trial` and `ms_panel`.
4. Run the sealed synthetic recall-under-capacity study.

In [ ]:
# [1]
MS_SEED, MS_ETA, MS_GATE = 6050, 0.20, 0.10
MS_DIMS, MS_LOADS, MS_REPEATS = (8, 16, 32), (0.5, 1, 2, 4, 8), 30

# [2]
def ms_unit(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def ms_decode(prediction, values):
    return np.argmax(prediction @ values.T, axis=1)

# [3]
def ms_trial(rng, repeat, d, load):
    n = max(4, round(load * d))
    keys = ms_unit(rng.standard_normal((n, d)))
    values = ms_unit(rng.standard_normal((n, d)))
    priority = np.zeros(n, dtype=bool)
    priority[rng.permutation(n)[:max(1, round(0.25 * n))]] = True

    scores = 32.0 * (keys @ keys.T)
    scores -= scores.max(axis=1, keepdims=True)
    weights = np.exp(scores); weights /= weights.sum(axis=1, keepdims=True)
    attention = weights @ values

    def delta(write_gate):
        state = np.zeros((d, d))
        for key, value, gate in zip(keys, values, write_gate, strict=True):
            state += MS_ETA * gate * np.outer(value - state @ key, key)
        return keys @ state.T

    plain = delta(np.ones(n))
    selective = delta(np.where(priority, 1.0, MS_GATE))
    truth = np.arange(n)
    correct = lambda pred: ms_decode(pred, values) == truth
    a_ok, d_ok, s_ok = correct(attention), correct(plain), correct(selective)
    ordinary = ~priority
    return [repeat, d, n, load, a_ok.mean(), ((attention-values)**2).mean(),
            d_ok.mean(), d_ok[priority].mean(), d_ok[ordinary].mean(),
            s_ok.mean(), s_ok[priority].mean(), s_ok[ordinary].mean()]

def ms_panel(tag, repeats):
    seeds = np.random.SeedSequence([MS_SEED, tag]).spawn(
        repeats * len(MS_DIMS) * len(MS_LOADS))
    rows, index = [], 0
    for repeat in range(repeats):
        for d in MS_DIMS:
            for load in MS_LOADS:
                rows.append(ms_trial(np.random.default_rng(seeds[index]),
                                     repeat, d, load))
                index += 1
    return np.asarray(rows)

# Tags 0 and 1 were development; tag 2 remained sealed until settings were fixed.
ms_development = np.vstack([ms_panel(tag, 5) for tag in (0, 1)])
assert np.all(ms_development[:, 4] == 1.0)
ms_endpoint = ms_panel(2, MS_REPEATS)
assert np.all(ms_endpoint[:, 4] == 1.0)

print("study_seed=6050; development_tags=(0, 1); sealed_endpoint_tag=2")
print("endpoint_trials=30 x 3 dimensions; d=(8, 16, 32); "
      "N/d=(0.5, 1, 2, 4, 8)")
print("fixed: eta=0.20; priority_fraction=0.25; ordinary_write_gate=0.10")
print("load  attention  Delta  gated-all  gated-priority  gated-ordinary")
# [4]
for load in MS_LOADS:
    block = ms_endpoint[ms_endpoint[:, 3] == load]
    means = block[:, [4, 6, 9, 10, 11]].mean(0)
    print(f"{load:>4.1f}   " + "  ".join(f"{value:.3f}" for value in means))

hard = ms_endpoint[ms_endpoint[:, 3] == 8]
priority_gain = hard[:, 10] - hard[:, 7]
ordinary_gain = hard[:, 11] - hard[:, 8]
print(f"N/d=8 gated-minus-Delta priority: {priority_gain.mean():+.3f} "
      f"(SD {priority_gain.std(ddof=1):.3f})")
print(f"N/d=8 gated-minus-Delta ordinary: {ordinary_gain.mean():+.3f} "
      f"(SD {ordinary_gain.std(ddof=1):.3f})")
print(f"attention top-1 failures=0/{int(ms_endpoint[:, 2].sum())}")
print(f"mean value MSE={ms_endpoint[:, 5].mean():.2e}; "
      f"max trial MSE={ms_endpoint[:, 5].max():.2e}")